# Automatic Question Answering and Intro to Agents

**Audience:** High school students  
**Format:** 1 notebook, hands-on class, about 90 minutes

In this class we will learn three core ideas:

1. **LLM**: a language model that can read instructions and generate text.
2. **Prompt**: the instructions and context we give to the model.
3. **Agent**: an LLM that can use tools to solve a task.

By the end, students will build:

- a basic automatic question answering system,
- a simple RAG agent,
- a simple SQL agent.

Reference ideas are adapted from LangChain documentation:

- ChatOpenAI: https://docs.langchain.com/oss/python/integrations/chat/openai
- RAG: https://docs.langchain.com/oss/python/langchain/rag
- SQL Agent: https://docs.langchain.com/oss/python/langchain/sql-agent


## 0. Introduction

### What is Automatic Question Answering?

Automatic Question Answering means we ask a computer a question in normal language, and the computer tries to answer clearly.

Example:

- Student asks: `What is photosynthesis?`
- System answers: `Photosynthesis is the process plants use to turn sunlight into energy.`

### Core components

**1. LLM**

The LLM is the brain that reads text and generates an answer.

**2. Prompt**

The prompt tells the model what role to play and how to answer.

**3. Agent**

An agent is an LLM plus tools. Instead of only guessing from memory, it can use outside help such as:

- search over documents,
- a database,
- a calculator,
- APIs.

A simple mental model:

- **LLM only** = answer with text generation
- **LLM + tools** = agent


## Class plan

- Part 1: Basic question answering with OpenAI API
- Part 2: Why tools matter
- Part 3: RAG agent
- Part 4: SQL agent
- Part 5: Wrap-up and discussion


In [ ]:
%pip install -qU langchain langchain-openai langchain-community

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

## 1. Basic automatic question answering

We start with the simplest version: ask an LLM a question and get an answer.

This is not an agent yet. It is only the model plus a prompt.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [ ]:
messages = [
    (
        "system",
        "You are a friendly teaching assistant for high school students. Answer clearly in 2 to 4 short sentences."
    ),
    ("human", "What is artificial intelligence?")
]

response = llm.invoke(messages)
print(response.content)

### Why this works

- The **system message** defines the role.
- The **human message** gives the question.
- The model returns a text answer.

This is the first version of an automatic question answering system.

In [ ]:
def ask_basic_qa(question: str):
    messages = [
        (
            "system",
            "You are a helpful teacher. Give simple answers for high school students."
        ),
        ("human", question),
    ]
    return llm.invoke(messages).content

print(ask_basic_qa("Why do we need prompts?"))

### Mini activity

Try changing the question:

- `What is machine learning?`
- `Explain an agent in simple words.`
- `What is the difference between data and information?`

Also try changing the system prompt to make the answer:

- shorter,
- more formal,
- more fun.


## 2. From LLM to Agent

A plain LLM answers using its training and the prompt.

But sometimes that is not enough.

Example problems:

- It may not know your private class notes.
- It may need fresh information.
- It may need to query a database.

So we give the LLM **tools**.

When an LLM can decide to use tools, we often call it an **agent**.

## 2.1 RAG Agent

**RAG** stands for **Retrieval-Augmented Generation**.

Idea:

1. Store useful documents.
2. Retrieve the most relevant text for a question.
3. Let the model answer using that retrieved context.

This is useful when the answer should come from specific materials instead of only from the model's memory.

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Bangkok is the capital city of Thailand."),
    Document(page_content="Chiang Mai is known for mountains, temples, and northern culture."),
    Document(page_content="Phuket is famous for beaches and tourism."),
    Document(page_content="Ayutthaya was once the capital of the Kingdom of Siam."),
]

docs

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(docs)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("Which place in Thailand is famous for beaches?")
retrieved_docs

### Turn retrieval into a tool

Now we make a tool that the model can call.

In [ ]:
from typing import List

def retrieve_context(question: str) -> str:
    """Search the class knowledge base and return relevant context."""
    matches = retriever.invoke(question)
    return "\n\n".join(doc.page_content for doc in matches)

print(retrieve_context("What city is the capital of Thailand?"))

In [ ]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model=llm,
    tools=[retrieve_context],
    system_prompt=(
        "You are a question answering assistant for students. "
        "Use the retrieve_context tool when the question needs facts from the class notes."
    ),
)

To keep the class output simple, we will only show the final answer instead of the full tool trace.

In [ ]:
def show_final_answer(agent_result):
    for message in reversed(agent_result["messages"]):
        content = getattr(message, "content", "")
        if content:
            return content
    return "No final answer was returned."

In [ ]:
result = rag_agent.invoke(
    {
        "messages": [
            ("human", "Which city is the capital of Thailand?")
        ]
    }
)

print(show_final_answer(result))

### What happened?

- The agent received the question.
- It had access to one tool: `retrieve_context`.
- The tool searched the small document collection.
- The model used the retrieved text to answer.

This is a very small RAG agent.

### Student challenge

Add two new `Document(...)` notes to the knowledge base and ask a new question based on your new facts.

## 2.2 SQL Agent

A SQL agent answers questions by reading a database.

Instead of searching text documents, it can inspect tables and run SQL queries.

For a beginner class, we will use a tiny SQLite database created inside the notebook.

In [ ]:
import sqlite3

conn = sqlite3.connect("school.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute(
    """
    CREATE TABLE students (
        id INTEGER PRIMARY KEY,
        name TEXT,
        grade_level INTEGER,
        math_score INTEGER,
        science_score INTEGER
    )
    """
)

rows = [
    (1, "Mali", 10, 88, 91),
    (2, "Niran", 10, 72, 84),
    (3, "Anan", 11, 95, 89),
    (4, "Suda", 11, 81, 93),
    (5, "Pim", 12, 90, 87),
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?)", rows)
conn.commit()

cursor.execute("SELECT * FROM students")
cursor.fetchall()

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///school.db")
print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())
print(db.run("SELECT name, math_score FROM students ORDER BY math_score DESC LIMIT 3;"))

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()
sql_tools

In [ ]:
sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=(
        "You are a helpful data assistant. "
        "Use the SQL tools to answer questions about the school database. "
        "Do not make up data."
    ),
)

In [ ]:
sql_result = sql_agent.invoke(
    {
        "messages": [
            ("human", "Which student has the highest science score?")
        ]
    }
)

print(show_final_answer(sql_result))

### Why this is different from RAG

- **RAG agent** uses text documents.
- **SQL agent** uses tables and SQL queries.

Both are agents because both use tools.

## 3. Compare the three levels

| Level | What it can do | Example |
|---|---|---|
| LLM only | Answer from prompt and general knowledge | Basic Q&A |
| RAG agent | Read retrieved notes or documents | Answer from class notes |
| SQL agent | Query structured data in tables | Answer from student score database |


## 4. Wrap-up

Today we learned:

- A **prompt** controls how the model answers.
- An **LLM** can do basic question answering.
- An **agent** is an LLM with tools.
- A **RAG agent** uses documents.
- A **SQL agent** uses databases.

A useful final idea:

- If your information is in text, use RAG.
- If your information is in tables, use SQL.


## 5. Optional extension ideas

If there is extra time, students can try one of these:

1. Add more documents to the RAG example.
2. Add a new table column such as `english_score`.
3. Ask the SQL agent for averages, highest scores, or students by grade.
4. Change the system prompt so the assistant answers in Thai or English.


## 6. How this can become a chatbot

What we built today is mostly a **question answering system**.

A **chatbot** is the next step. It usually adds:

- **conversation history** so it remembers earlier messages,
- **a chat interface** so users can talk back and forth,
- **tools** such as RAG or SQL when it needs outside information.

A simple way to explain it is:

- **Q&A system** = answer one question well
- **Chatbot** = answer many connected questions in a conversation

So a chatbot can be built from the same ideas we learned today, with memory and a conversation flow added on top.